# Bivariate & Correlation Analysis

Deep-dive correlation notebook referenced by **[EDA_Governance.ipynb](EDA_Governance.ipynb)**.  
Covers: pairwise correlations, heatmaps, scatter matrices, categorical cross-tabs, and target leakage checks.

In [ ]:
# --- Setup: imports and theme ---
import sys, os
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy import stats

sys.path.insert(0, os.path.abspath('../../src'))
from elocal_analysis.elocal_theme import (
    set_elocal_theme, ELOCAL_PALETTE, ELOCAL_BLUE, ELOCAL_ORANGE, ELOCAL_GREEN,
    ELOCAL_SEQ_CMAP
)

set_elocal_theme()
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 100)

In [ ]:
# --- Load your dataset here ---
# DATA_PATH = os.path.join('../../data/raw/', 'your_file.csv')
# df = pd.read_csv(DATA_PATH)
# df.head()

## Correlation Matrix

In [ ]:
# --- Pearson correlation matrix (numeric columns) ---
def correlation_matrix(df: pd.DataFrame, method: str = 'pearson') -> pd.DataFrame:
    """Compute pairwise correlation matrix for numeric columns.
    method: 'pearson', 'spearman', or 'kendall'."""
    corr = df.select_dtypes(include='number').corr(method=method)
    return corr

# correlation_matrix(df)

In [ ]:
# --- Top-N strongest pairwise correlations ---
def top_correlations(df: pd.DataFrame, n: int = 20,
                     method: str = 'pearson') -> pd.DataFrame:
    """Return the top-n absolute pairwise correlations (no duplicates, no self-pairs)."""
    corr = df.select_dtypes(include='number').corr(method=method)
    pairs = (
        corr.where(np.triu(np.ones(corr.shape), k=1).astype(bool))
        .stack()
        .reset_index()
    )
    pairs.columns = ['feature_1', 'feature_2', 'correlation']
    pairs['abs_corr'] = pairs['correlation'].abs()
    return pairs.sort_values('abs_corr', ascending=False).head(n).drop(columns='abs_corr')

# top_correlations(df)

## Correlation Heatmap

In [ ]:
# --- Full correlation heatmap (lower triangle) ---
def plot_correlation_heatmap(df: pd.DataFrame, method: str = 'pearson',
                              figsize: tuple = (16, 12)) -> None:
    """Lower-triangle heatmap with annotations."""
    numeric_df = df.select_dtypes(include='number')
    if numeric_df.shape[1] < 2:
        print('Need at least 2 numeric columns for a heatmap.')
        return
    corr = numeric_df.corr(method=method)
    mask = np.triu(np.ones_like(corr, dtype=bool), k=1)
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(corr, mask=mask, annot=True, fmt='.2f',
                cmap='coolwarm', center=0, square=True,
                linewidths=0.5, ax=ax,
                cbar_kws={'shrink': 0.8, 'label': f'{method.title()} r'})
    ax.set_title(f'{method.title()} Correlation Heatmap', pad=20)
    plt.tight_layout()
    plt.show()

# plot_correlation_heatmap(df)

In [ ]:
# --- Clustered correlation heatmap (reorders by similarity) ---
def plot_clustered_heatmap(df: pd.DataFrame, method: str = 'pearson') -> None:
    """Seaborn clustermap — rows and columns reordered by hierarchical clustering."""
    numeric_df = df.select_dtypes(include='number')
    if numeric_df.shape[1] < 2:
        print('Need at least 2 numeric columns.')
        return
    corr = numeric_df.corr(method=method)
    g = sns.clustermap(corr, annot=True, fmt='.2f', cmap='coolwarm',
                       center=0, linewidths=0.5, figsize=(14, 12),
                       cbar_kws={'label': f'{method.title()} r'})
    g.fig.suptitle(f'Clustered {method.title()} Correlation', y=1.02,
                   fontsize=18, fontweight='bold')
    plt.show()

# plot_clustered_heatmap(df)

## Scatter & Pair Plots

In [ ]:
# --- Scatter plot between two columns ---
def plot_scatter(df: pd.DataFrame, x: str, y: str,
                 hue: str = None) -> None:
    """Scatter plot with optional hue grouping and regression line."""
    fig, ax = plt.subplots(figsize=(16, 9))
    sns.scatterplot(data=df, x=x, y=y, hue=hue,
                    palette=ELOCAL_PALETTE, alpha=0.6, ax=ax)
    sns.regplot(data=df, x=x, y=y, scatter=False,
                color=ELOCAL_ORANGE, ax=ax, line_kws={'linewidth': 2})
    r, p = stats.pearsonr(df[x].dropna(), df[y].dropna())
    ax.set_title(f'{y} vs {x}  (r={r:.3f}, p={p:.3e})')
    plt.tight_layout()
    plt.show()

# plot_scatter(df, 'col_x', 'col_y')

In [ ]:
# --- Pair plot for selected columns ---
def plot_pairplot(df: pd.DataFrame, cols: list = None,
                  hue: str = None) -> None:
    """Seaborn pair plot for a subset of columns (defaults to all numeric)."""
    if cols is None:
        cols = df.select_dtypes(include='number').columns.tolist()
    if len(cols) > 8:
        print(f'{len(cols)} columns — using first 8 to keep plot readable.')
        cols = cols[:8]
    subset = cols + ([hue] if hue and hue not in cols else [])
    g = sns.pairplot(df[subset].dropna(), hue=hue,
                     palette=ELOCAL_PALETTE, diag_kind='kde',
                     plot_kws={'alpha': 0.5})
    g.fig.suptitle('Pair Plot', y=1.02, fontsize=18, fontweight='bold')
    plt.show()

# plot_pairplot(df, ['col_a', 'col_b', 'col_c'])

## Categorical Cross-Tabs

In [ ]:
# --- Cross-tabulation between two categorical columns ---
def crosstab_analysis(df: pd.DataFrame, col_a: str, col_b: str,
                      normalize: str = 'index') -> pd.DataFrame:
    """Cross-tab with optional normalisation ('index', 'columns', 'all', or None)."""
    ct = pd.crosstab(df[col_a], df[col_b], normalize=normalize)
    return ct

# crosstab_analysis(df, 'cat_col_1', 'cat_col_2')

In [ ]:
# --- Heatmap of cross-tab frequencies ---
def plot_crosstab_heatmap(df: pd.DataFrame, col_a: str, col_b: str) -> None:
    """Heatmap of raw counts between two categorical columns."""
    ct = pd.crosstab(df[col_a], df[col_b])
    fig, ax = plt.subplots(figsize=(16, 9))
    sns.heatmap(ct, annot=True, fmt='d', cmap=ELOCAL_SEQ_CMAP,
                linewidths=0.5, ax=ax)
    ax.set_title(f'Cross-Tab: {col_a} × {col_b}')
    plt.tight_layout()
    plt.show()

# plot_crosstab_heatmap(df, 'cat_col_1', 'cat_col_2')

In [ ]:
# --- Chi-squared test of independence ---
def chi_squared_test(df: pd.DataFrame, col_a: str, col_b: str) -> dict:
    """Run chi-squared test on two categorical columns. Returns stat, p-value, dof."""
    ct = pd.crosstab(df[col_a], df[col_b])
    chi2, p, dof, expected = stats.chi2_contingency(ct)
    result = {'chi2': round(chi2, 4), 'p_value': p, 'dof': dof,
              'significant_at_05': p < 0.05}
    print(f"Chi² = {chi2:.4f}, p = {p:.3e}, dof = {dof}  →  "
          f"{'SIGNIFICANT' if p < 0.05 else 'not significant'} at α=0.05")
    return result

# chi_squared_test(df, 'cat_col_1', 'cat_col_2')

## Target Leakage Detection

In [ ]:
# --- Target leakage: high-correlation flag for numeric target ---
def target_leakage_check(df: pd.DataFrame, target_col: str,
                          threshold: float = 0.95) -> pd.DataFrame:
    """Flag features with |correlation| >= threshold to the target."""
    numeric_df = df.select_dtypes(include='number')
    if target_col not in numeric_df.columns:
        print(f'{target_col} is not numeric — cannot compute correlation.')
        return pd.DataFrame()
    corr = numeric_df.corr()[target_col].drop(target_col).abs().sort_values(ascending=False)
    leakers = corr[corr >= threshold]
    if leakers.empty:
        print(f'No features above {threshold} correlation with "{target_col}".')
    else:
        print(f'⚠ POTENTIAL LEAKAGE — features with |corr| >= {threshold} to "{target_col}":')
    return leakers.reset_index().rename(columns={'index': 'feature', target_col: 'abs_corr'})

# target_leakage_check(df, 'target_column')

In [ ]:
# --- Target leakage: bar chart of feature correlations with target ---
def plot_target_correlations(df: pd.DataFrame, target_col: str,
                              top_n: int = 20) -> None:
    """Horizontal bar chart of absolute correlations with the target column."""
    numeric_df = df.select_dtypes(include='number')
    if target_col not in numeric_df.columns:
        print(f'{target_col} is not numeric.')
        return
    corr = numeric_df.corr()[target_col].drop(target_col).sort_values(key=abs, ascending=False).head(top_n)
    colors = [ELOCAL_ORANGE if abs(v) >= 0.95 else ELOCAL_BLUE for v in corr.values]
    fig, ax = plt.subplots(figsize=(16, max(4, len(corr) * 0.4)))
    sns.barplot(x=corr.values, y=corr.index, palette=colors, ax=ax)
    ax.axvline(0, color='grey', linewidth=0.8)
    ax.set_xlabel('Correlation')
    ax.set_title(f'Top {top_n} Feature Correlations with "{target_col}"\n(orange = potential leakage ≥ 0.95)')
    plt.tight_layout()
    plt.show()

# plot_target_correlations(df, 'target_column')

## Numeric vs Categorical Relationships

In [ ]:
# --- Box plot: numeric column grouped by a categorical column ---
def plot_numeric_by_category(df: pd.DataFrame, numeric_col: str,
                              cat_col: str, top_n: int = 10) -> None:
    """Box plot of a numeric column grouped by top_n categories."""
    top_cats = df[cat_col].value_counts().head(top_n).index
    subset = df[df[cat_col].isin(top_cats)]
    fig, ax = plt.subplots(figsize=(16, 8))
    sns.boxplot(data=subset, x=cat_col, y=numeric_col,
                palette=ELOCAL_PALETTE, ax=ax)
    ax.set_title(f'{numeric_col} by {cat_col} (top {top_n} categories)')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=45, ha='right')
    plt.tight_layout()
    plt.show()

# plot_numeric_by_category(df, 'revenue', 'category')

In [ ]:
# --- ANOVA: numeric column across groups of a categorical column ---
def anova_test(df: pd.DataFrame, numeric_col: str,
               cat_col: str) -> dict:
    """One-way ANOVA to test if group means differ significantly."""
    groups = [g[numeric_col].dropna().values
              for _, g in df.groupby(cat_col) if len(g[numeric_col].dropna()) > 1]
    if len(groups) < 2:
        print('Need at least 2 groups with data.')
        return {}
    f_stat, p = stats.f_oneway(*groups)
    result = {'F_statistic': round(f_stat, 4), 'p_value': p,
              'significant_at_05': p < 0.05}
    print(f"ANOVA: F={f_stat:.4f}, p={p:.3e}  →  "
          f"{'SIGNIFICANT' if p < 0.05 else 'not significant'} at α=0.05")
    return result

# anova_test(df, 'revenue', 'category')

---

### Quick-Run Template

Uncomment once `df` is loaded.

In [ ]:
# --- Quick-run: execute all correlation checks ---
# print('=== CORRELATION MATRIX ===')
# display(top_correlations(df))
#
# print('\n=== HEATMAPS ===')
# plot_correlation_heatmap(df)
# plot_clustered_heatmap(df)
#
# print('\n=== TARGET LEAKAGE ===')
# target_leakage_check(df, 'target_column')
# plot_target_correlations(df, 'target_column')